# Vehicle Safety Complaint Triage: Is a Fine-Tuned Transformer Worth It?

## What we are building
Car owners send safety complaints to the US government (NHTSA). Thousands arrive every month, and engineers can't read them all.

We build a **triage system** that, for every complaint:
1. Tags **which car part failed** (brakes, airbags, engine…)
2. Flags **dangerous** complaints (crash, fire, injury, death)
3. Finds **similar past complaints**
4. Writes a short **"emerging issues" report** for engineers

**Analogy:** a hospital triage nurse, but for cars. Sort patients by what's wrong and who needs help first.

## The experiment
> Does a fine-tuned DistilBERT beat a simple TF-IDF + logistic regression model by enough to justify being slower and more expensive?

## The pipeline
```
Download → clean → split by time → simple model → fine-tuned model → compare
   → error analysis → track (MLflow) → similar search → emerging issues (LLM)
   → drift check → API → demo app
```

## How to use this notebook
- **Runtime → Change runtime type → T4 GPU**
- Run cells **top to bottom, one at a time**. Read the note above each cell first.
- Total time: about **60–90 minutes**.

## Concepts cheat sheet
| Concept | Simple meaning |
|---|---|
| Classification | Putting each item into a bucket ("brakes", "airbags"…) |
| Class imbalance | Some buckets are huge, some tiny. Dangerous complaints are rare. |
| TF-IDF | Counts words, but gives more weight to rare, telling words like "airbag" than common ones like "the" |
| Logistic regression | A simple model that learns how much each word pushes toward each bucket |
| DistilBERT | A small transformer that reads words **in context** ("brake light" vs "brake pedal") |
| Fine-tuning | Giving a pretrained model extra lessons on our task |
| Time-based split | Train on the past, test on the future, like real life |
| Macro-F1 | Average score across buckets, so rare buckets count as much as big ones |
| PR-AUC | How well we rank dangerous complaints above safe ones (good for rare events) |
| Recall | Of all truly dangerous complaints, how many did we catch? |
| MLflow | A lab notebook that records every experiment automatically |
| Drift | The world changes, so new data stops looking like training data |
| PSI | A number that measures how much a distribution has shifted |
| API | A "vending machine" for your model: send text in, get predictions out |

# Part 0 - Setup

### 0.1 Check the GPU
You should see a **Tesla T4**. The GPU is only needed for Part 5 (fine-tuning) and Part 9 (LLM). Everything else runs on CPU.

In [1]:
!nvidia-smi

Sat Nov 14 22:57:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 0.2 Install libraries
- `transformers`, `datasets`, `accelerate` → fine-tune DistilBERT
- `sentence-transformers`, `chromadb` → similar-complaint search
- `mlflow` → experiment tracking
- `fastapi`, `uvicorn` → serve the model as an API
- `gradio` → demo app

`scikit-learn` and `pandas` are already in Colab. If Colab says **"restart session"**, click it, then continue from the next cell.

In [2]:
!pip install -q -U transformers datasets accelerate sentence-transformers chromadb mlflow fastapi uvicorn gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


### 0.3 Imports and settings
- `SEED` → same random shuffles every run, so results are repeatable
- `DEVICE` → use the GPU if there is one
- We set `report_to="none"` later in training instead of `WANDB_DISABLED` (newer libraries prefer it). Same goal: **no login popup**.

In [3]:
import os, re, csv, glob, json, time, random, zipfile, urllib.request
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

pd.set_option("display.max_colwidth", 200)
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

Using: cuda


### 0.4 (Optional) Save to Google Drive
Colab erases files when the session ends. Run this to keep models and results.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/nhtsa_triage"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/nhtsa_triage


# Part 1 - Get the data

NHTSA publishes every complaint since 1995 as free **tab-separated** text files, zipped in **5-year chunks** by the date NHTSA received them.

- `COMPLAINTS_RECEIVED_2020-2024.zip` → our **training and test** years
- `COMPLAINTS_RECEIVED_2025-2029.zip` → newer complaints, used as **"the future"** to check drift

If a download fails (file names can change), open the NHTSA flat-files download page in your browser, download the complaints zip yourself, and upload it into the `data/` folder (folder icon in the left sidebar).

In [5]:
BASE_URL = "https://static.nhtsa.gov/odi/ffdd/cmpl/"

def download(fname):
    path = f"data/{fname}"
    if os.path.exists(path):
        print("Already have", fname); return True
    try:
        urllib.request.urlretrieve(BASE_URL + fname, path)
        print("Downloaded", fname, f"({os.path.getsize(path)/1e6:.0f} MB)")
        return True
    except Exception as e:
        print("Not found:", fname, "→", e)
        return False

# 1) Training + test years (this one already worked)
download("COMPLAINTS_RECEIVED_2020-2024.zip")

# 2) The "future" chunk: try possible names until one works
for end in [2026, 2025, 2027, 2028, 2029]:
    if download(f"COMPLAINTS_RECEIVED_2025-{end}.zip"):
        break

print("Zip files in data/:", glob.glob("data/*.zip"))

Downloaded COMPLAINTS_RECEIVED_2020-2024.zip (76 MB)
Downloaded COMPLAINTS_RECEIVED_2025-2026.zip (44 MB)
Zip files in data/: ['data/COMPLAINTS_RECEIVED_2025-2026.zip', 'data/COMPLAINTS_RECEIVED_2020-2024.zip']


### 1.1 Read the files
The files have **no header row**, so we give the column names ourselves (from NHTSA's field list, `CMPL.txt`).

We only keep the columns we need:
- **Privacy:** we **drop** names, VINs, dealer phone numbers and addresses. A good habit, and a good interview point.
- `quoting=csv.QUOTE_NONE` → complaint text contains quote marks; this stops pandas from getting confused
- `on_bad_lines="skip"` → skip the few broken rows instead of crashing
- `dtype=str` → read everything as text first, convert later

In [6]:
COLS = ["CMPLID","ODINO","MFR_NAME","MAKETXT","MODELTXT","YEARTXT","CRASH","FAILDATE","FIRE",
        "INJURED","DEATHS","COMPDESC","CITY","STATE","VIN","DATEA","LDATE","MILES","OCCURENCES",
        "CDESCR","CMPL_TYPE","POLICE_RPT_YN","PURCH_DT","ORIG_OWNER_YN","ANTI_BRAKES_YN",
        "CRUISE_CONT_YN","NUM_CYLS","DRIVE_TRAIN","FUEL_SYS","FUEL_TYPE","TRANS_TYPE","VEH_SPEED",
        "DOT","TIRE_SIZE","LOC_OF_TIRE","TIRE_FAIL_TYPE","ORIG_EQUIP_YN","MANUF_DT","SEAT_TYPE",
        "RESTRAINT_TYPE","DEALER_NAME","DEALER_TEL","DEALER_CITY","DEALER_STATE","DEALER_ZIP",
        "PROD_TYPE","REPAIRED_YN","MEDICAL_ATTN","VEHICLES_TOWED_YN","STATE_OF_INCIDENT",
        "VEHICLE_OPERATOR"]

KEEP = ["ODINO","MAKETXT","MODELTXT","YEARTXT","CRASH","FIRE","INJURED","DEATHS",
        "COMPDESC","LDATE","CDESCR","PROD_TYPE"]

def read_zip(path):
    frames = []
    with zipfile.ZipFile(path) as z:
        for name in z.namelist():
            if name.lower().endswith(".txt"):
                with z.open(name) as fh:
                    frames.append(pd.read_csv(
                        fh, sep="\t", header=None, names=COLS, usecols=KEEP, index_col=False,
                        dtype=str, quoting=csv.QUOTE_NONE, encoding="latin-1",
                        on_bad_lines="skip"))
    return pd.concat(frames, ignore_index=True)

raw = pd.concat([read_zip(p) for p in sorted(glob.glob("data/*.zip"))], ignore_index=True)
print(raw.shape)
raw.head(3)

(633275, 12)


,ODINO,MAKETXT,MODELTXT,YEARTXT,CRASH,FIRE,INJURED,DEATHS,COMPDESC,LDATE,CDESCR,PROD_TYPE
0,11292384,HONDA,ACCORD,2018,N,N,0,0,SERVICE BRAKES,20200101,"DRIVING AT THE HIGHWAY, CAR SUDDENLY SLOW DOWN FROM 70MPH TO 40-50MPH,THERE WERE ANY CAR IN FRONT OF ME! I HAVE NOTHING ON MY WINDSHIELD! THERE WEREN'T ANY STOP OR TRAFFIC SIGN! SO HONDA STOP SAYI...",V
1,11292384,HONDA,ACCORD,2018,N,N,0,0,ELECTRICAL SYSTEM,20200101,"DRIVING AT THE HIGHWAY, CAR SUDDENLY SLOW DOWN FROM 70MPH TO 40-50MPH,THERE WERE ANY CAR IN FRONT OF ME! I HAVE NOTHING ON MY WINDSHIELD! THERE WEREN'T ANY STOP OR TRAFFIC SIGN! SO HONDA STOP SAYI...",V
2,11292384,HONDA,ACCORD,2018,N,N,0,0,ENGINE,20200101,"DRIVING AT THE HIGHWAY, CAR SUDDENLY SLOW DOWN FROM 70MPH TO 40-50MPH,THERE WERE ANY CAR IN FRONT OF ME! I HAVE NOTHING ON MY WINDSHIELD! THERE WEREN'T ANY STOP OR TRAFFIC SIGN! SO HONDA STOP SAYI...",V


### 1.2 Clean and create labels
Each row becomes something the model can learn from.

- `PROD_TYPE == "V"` → keep **vehicles** only (drop tires, child seats, equipment)
- `date` → when NHTSA received it (we split by this later)
- `text` → lowercase complaint; remove the `TL*` prefix NHTSA adds to many descriptions
- `component` → `"SERVICE BRAKES, HYDRAULIC:FOUNDATION COMPONENTS"` becomes `"SERVICE BRAKES, HYDRAULIC"` (keep the part before `:`, the main system)
- `danger` → **1** if crash **or** fire **or** anyone injured **or** died, else **0**

In [7]:
df = raw[raw.PROD_TYPE == "V"].copy()

df["date"] = pd.to_datetime(df.LDATE, format="%Y%m%d", errors="coerce")
df["text"] = (df.CDESCR.fillna("").str.lower()
                .str.replace(r"^\s*tl\*\s*", "", regex=True)
                .str.replace(r"\s+", " ", regex=True).str.strip())
df["component"] = df.COMPDESC.fillna("UNKNOWN").str.split(":").str[0].str.strip()

for c in ["INJURED", "DEATHS"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["danger"] = ((df.CRASH == "Y") | (df.FIRE == "Y") | (df.INJURED > 0) | (df.DEATHS > 0)).astype(int)

print(df.shape)
df[["ODINO", "date", "MAKETXT", "MODELTXT", "component", "danger", "text"]].head(3)

(624568, 16)


,ODINO,date,MAKETXT,MODELTXT,component,danger,text
0,11292384,2020-01-01,HONDA,ACCORD,SERVICE BRAKES,0,"driving at the highway, car suddenly slow down from 70mph to 40-50mph,there were any car in front of me! i have nothing on my windshield! there weren't any stop or traffic sign! so honda stop sayi..."
1,11292384,2020-01-01,HONDA,ACCORD,ELECTRICAL SYSTEM,0,"driving at the highway, car suddenly slow down from 70mph to 40-50mph,there were any car in front of me! i have nothing on my windshield! there weren't any stop or traffic sign! so honda stop sayi..."
2,11292384,2020-01-01,HONDA,ACCORD,ENGINE,0,"driving at the highway, car suddenly slow down from 70mph to 40-50mph,there were any car in front of me! i have nothing on my windshield! there weren't any stop or traffic sign! so honda stop sayi..."


### 1.3 One complaint = one row, one label
One complaint (`ODINO`) can mention **several** parts, so it appears in several rows.

To keep the task simple and fair:
- Keep complaints that name **exactly one** main component (one clear label)
- Keep **one row** per complaint
- Drop unclear labels (`UNKNOWN OR OTHER`) and very short texts

**Analogy:** for a sorting game, only use toys that clearly belong in one box.

In [8]:
n_parts = df.groupby("ODINO")["component"].nunique()
single = n_parts[n_parts == 1].index

clean = (df[df.ODINO.isin(single)]
           .drop_duplicates("ODINO")
           .dropna(subset=["date"]))
clean = clean[~clean.component.str.contains("UNKNOWN", na=False)]
clean = clean[clean.text.str.split().str.len() >= 20]

print("Rows:", len(df), "→ clean complaints:", len(clean))
print("Date range:", clean.date.min().date(), "to", clean.date.max().date())

Rows: 624568 → clean complaints: 247376
Date range: 2020-01-01 to 2026-11-12


# Part 2 — Explore the data (EDA)

Always look before you model.

### 2.1 How common is each component?
Some parts get far more complaints than others. That's **class imbalance**.

In [9]:
comp_counts = clean.component.value_counts()
print("Number of components:", len(comp_counts))
comp_counts.head(15)

Number of components: 49


,count
component,
ENGINE,44216
POWER TRAIN,33070
ELECTRICAL SYSTEM,25541
STEERING,21429
SERVICE BRAKES,14733
AIR BAGS,14654
STRUCTURE,12980
FUEL/PROPULSION SYSTEM,10340
ENGINE AND ENGINE COOLING,7747


### 2.2 How rare are dangerous complaints?
If only ~10% are dangerous, a model that always says "safe" is **90% accurate** but useless. That's why we use **recall** and **PR-AUC**, not accuracy, for this task.

In [10]:
print("Danger rate:", round(clean.danger.mean(), 3))
clean.groupby(clean.date.dt.year).danger.agg(["mean", "size"]).round(3)

Danger rate: 0.052


,mean,size
date,,
2020,0.061,33208
2021,0.065,28769
2022,0.059,31618
2023,0.051,36202
2024,0.039,39028
2025,0.043,40845
2026,0.052,37706


### 2.3 How long are complaints?
DistilBERT reads at most **512 tokens**. We'll use 256 to be faster. Long complaints get cut, same idea as chunking in PubMedQA.

In [11]:
clean.text.str.split().str.len().describe().round(0)

,text
count,247376.0
mean,106.0
std,69.0
min,20.0
25%,60.0
50%,90.0
75%,131.0
max,430.0


# Part 3 — Split by time (no peeking!)

In real life, you train on **past** complaints and predict **new** ones. So:
- **Train:** received 2021–2023
- **Test:** received 2024
- **Future:** received 2025 or later (for drift checks)

**Random splits would cheat:** the same recall wave or defect would appear in train and test, making scores look better than reality.

**Analogy:** study last year's papers, then take this year's exam.

We also:
- Pick the **top 10 components using train only** (choosing with test data is also peeking)
- **Sample** to keep training time short on a free GPU (raise these later for final numbers)
- Hold out **10% of train** as **validation**, used to choose the danger threshold

In [12]:
TRAIN_YEARS = [2021, 2022, 2023]
TEST_YEAR   = 2024
TOP_K       = 10
N_TRAIN, N_TEST, N_FUTURE = 60_000, 15_000, 15_000

year = clean.date.dt.year
train_all  = clean[year.isin(TRAIN_YEARS)]
test_all   = clean[year == TEST_YEAR]
future_all = clean[year > TEST_YEAR]

TOP_COMPONENTS = train_all.component.value_counts().head(TOP_K).index.tolist()

def keep_top(d):
    return d[d.component.isin(TOP_COMPONENTS)]

def sample(d, n):
    return d.sample(min(n, len(d)), random_state=SEED).reset_index(drop=True)

train_full = sample(keep_top(train_all), N_TRAIN)
test_df    = sample(keep_top(test_all),  N_TEST)
future_df  = sample(keep_top(future_all), N_FUTURE)

val_df   = train_full.sample(frac=0.10, random_state=SEED)
train_df = train_full.drop(val_df.index).reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("Top components:", TOP_COMPONENTS)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | future {len(future_df)}")
if len(future_df) == 0:
    print("No 2025+ data found; drift checks in Part 8 will use the test set instead.")

Top components: ['ENGINE', 'POWER TRAIN', 'ELECTRICAL SYSTEM', 'STEERING', 'AIR BAGS', 'SERVICE BRAKES', 'STRUCTURE', 'FUEL/PROPULSION SYSTEM', 'FORWARD COLLISION AVOIDANCE', 'VISIBILITY/WIPER']
train 54000 | val 6000 | test 15000 | future 15000


### 3.1 Leakage check
Make sure no complaint ID appears in both train and test. Should print **0**.

In [13]:
print("Overlap train/test:", len(set(train_df.ODINO) & set(test_df.ODINO)))
print("Overlap train/val:",  len(set(train_df.ODINO) & set(val_df.ODINO)))

Overlap train/test: 0
Overlap train/val: 0


### 3.2 Label encoding
Models need numbers, not words. `label2id` turns `"AIR BAGS"` into a number like `0`; `id2label` turns it back.

In [14]:
label2id = {c: i for i, c in enumerate(sorted(TOP_COMPONENTS))}
id2label = {i: c for c, i in label2id.items()}

for d in (train_df, val_df, test_df, future_df):
    d["comp_id"] = d.component.map(label2id)

pd.Series(label2id)

,0
AIR BAGS,0
ELECTRICAL SYSTEM,1
ENGINE,2
FORWARD COLLISION AVOIDANCE,3
FUEL/PROPULSION SYSTEM,4
POWER TRAIN,5
SERVICE BRAKES,6
STEERING,7
STRUCTURE,8
VISIBILITY/WIPER,9


# Part 4 — Simple baseline: TF-IDF + Logistic Regression

**The bicycle model.** Cheap, fast, often good enough. Every fancy model must beat this.

How it works:
1. **TF-IDF** turns each complaint into word scores. Rare, telling words ("airbag", "stall") score high; common words ("the", "car") score low.
2. `ngram_range=(1,2)` → also uses **2-word phrases** like "brake pedal", "check engine"
3. **Logistic regression** learns how much each word pushes toward each bucket
4. `class_weight="balanced"` → gives rare buckets more attention, so the model doesn't ignore them

We build **two** models:
- **Task A:** which component? (10 buckets)
- **Task B:** dangerous or not? (2 buckets)

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def make_tfidf_lr():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)),
        ("lr",    LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")),
    ])

t0 = time.time()
comp_lr = make_tfidf_lr().fit(train_df.text, train_df.comp_id)
print(f"Component model trained in {time.time()-t0:.0f}s")

t0 = time.time()
danger_lr = make_tfidf_lr().fit(train_df.text, train_df.danger)
print(f"Danger model trained in {time.time()-t0:.0f}s")

Component model trained in 131s
Danger model trained in 18s


### 4.1 Scoring functions
- **Component:** `accuracy` and **`macro_f1`** (every bucket counts equally, so rare parts matter)
- **Danger:** **`pr_auc`** (ranking quality, no threshold needed), plus `recall`, `precision`, `f1` at a chosen threshold

**Why recall matters most for danger:** missing a real crash is far worse than a false alarm that an engineer quickly dismisses.

In [16]:
from sklearn.metrics import (accuracy_score, f1_score, average_precision_score,
                             precision_score, recall_score, classification_report)

def score_component(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}

def score_danger(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {"pr_auc":    average_precision_score(y_true, prob),
            "recall":    recall_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0),
            "f1":        f1_score(y_true, pred, zero_division=0)}

### 4.2 Pick the danger threshold on validation
The model gives a **probability** (0 to 1). We must choose a cut-off for "flag it".

We pick the **highest threshold that still catches at least 80% of dangerous complaints** (recall ≥ 0.80), using **validation**, not test. Choosing with test data would be peeking.

**Analogy:** setting a smoke alarm's sensitivity in a practice room, not during the real fire drill.

In [17]:
TARGET_RECALL = 0.80

def pick_threshold(y_true, prob, target=TARGET_RECALL):
    best = 0.05
    for t in np.arange(0.05, 0.96, 0.01):
        if recall_score(y_true, (prob >= t).astype(int), zero_division=0) >= target:
            best = t
    return round(float(best), 2)

danger_threshold_lr = pick_threshold(val_df.danger, danger_lr.predict_proba(val_df.text)[:, 1])
print("Chosen threshold (TF-IDF):", danger_threshold_lr)

Chosen threshold (TF-IDF): 0.56


### 4.3 Test the baseline (and the "dumb" baseline)
- **Dumb component model:** always guess the most common part
- **Dumb danger model:** a random score; its PR-AUC equals the danger rate
- `ms_per_pred` → how long one prediction takes. Speed matters when you deploy.

In [18]:
def time_per_pred(fn, texts, n=1000):
    sample_texts = list(texts[:n])
    t0 = time.time(); fn(sample_texts)
    return (time.time() - t0) / len(sample_texts) * 1000

most_common = train_df.comp_id.mode()[0]
results = {}

results["dumb baseline"] = {
    **score_component(test_df.comp_id, np.full(len(test_df), most_common)),
    **score_danger(test_df.danger, np.full(len(test_df), test_df.danger.mean())),
    "ms_per_pred": 0.0,
}

comp_pred_lr = comp_lr.predict(test_df.text)
danger_prob_lr = danger_lr.predict_proba(test_df.text)[:, 1]
results["TF-IDF + LogReg"] = {
    **score_component(test_df.comp_id, comp_pred_lr),
    **score_danger(test_df.danger, danger_prob_lr, danger_threshold_lr),
    "ms_per_pred": time_per_pred(lambda t: (comp_lr.predict(t), danger_lr.predict_proba(t)), test_df.text),
}

pd.DataFrame(results).T.round(3)

,accuracy,macro_f1,pr_auc,recall,precision,f1,ms_per_pred
dumb baseline,0.234,0.038,0.041,0.000,0.000,0.000,0.000
TF-IDF + LogReg,0.891,0.885,0.830,0.763,0.796,0.779,0.337


### 4.4 Per-component report
Which parts are easy, which are hard? Look at the **f1-score** column.

In [19]:
print(classification_report(test_df.comp_id, comp_pred_lr,
                            target_names=[id2label[i] for i in range(len(id2label))], digits=3))

                             precision    recall  f1-score   support

                   AIR BAGS      0.934     0.957     0.945      1267
          ELECTRICAL SYSTEM      0.804     0.807     0.805      1820
                     ENGINE      0.903     0.897     0.900      3504
FORWARD COLLISION AVOIDANCE      0.867     0.871     0.869       479
     FUEL/PROPULSION SYSTEM      0.790     0.802     0.796       865
                POWER TRAIN      0.904     0.884     0.894      2588
             SERVICE BRAKES      0.913     0.894     0.903      1216
                   STEERING      0.972     0.964     0.968      1771
                  STRUCTURE      0.873     0.909     0.890      1072
           VISIBILITY/WIPER      0.860     0.895     0.877       418

                   accuracy                          0.891     15000
                  macro avg      0.882     0.888     0.885     15000
               weighted avg      0.892     0.891     0.891     15000



# Part 5 — Fine-tune DistilBERT (the heart of the project)

**The car model.** DistilBERT reads words **in context**: it knows "brake light" (electrical) is different from "brake pedal" (brakes), something TF-IDF only partly catches.

Fair experiment: **same train data, same test data, same metrics.** Only the model changes.

How fine-tuning works:
1. Start from DistilBERT, already trained on lots of general English
2. Add a small "decision layer" on top with one output per bucket
3. Show it our complaints + labels; it adjusts slightly each batch
4. One full pass over the data = **one epoch**

Settings:
- `MAX_LEN = 256` → read the first 256 tokens of each complaint
- `fp16=True` → half-size numbers, faster on a T4
- `report_to="none"` → no Weights & Biases login popup
- Time: about **5–10 minutes per model** on a T4

In [22]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
import math

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 32
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(texts, labels):
    ds = Dataset.from_dict({"text": list(texts), "label": [int(x) for x in labels]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=["text"])

def train_bert(texts, labels, num_labels, out_dir, epochs=1, id2lab=None):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels,
        id2label=id2lab, label2id={v: k for k, v in id2lab.items()} if id2lab else None)

    total_steps  = math.ceil(len(texts) / BATCH_SIZE) * epochs
    warmup_steps = int(0.1 * total_steps)          # same as warmup_ratio=0.1

    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE, learning_rate=5e-5,
        warmup_steps=warmup_steps, weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        logging_steps=100, save_strategy="no", report_to="none", seed=SEED)

    trainer = Trainer(model=model, args=args,
                      train_dataset=to_dataset(texts, labels),
                      data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    return model

### 5.1 Train the component model

In [23]:
EPOCHS = 1
comp_bert = train_bert(train_df.text, train_df.comp_id, num_labels=len(label2id),
                       out_dir="models/comp_bert", epochs=EPOCHS, id2lab=id2label)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/54000 [00:00<?, ? examples/s]

Step,Training Loss
100,2.093965
200,0.815120
300,0.525990
400,0.485419
500,0.429282
600,0.395227
700,0.375377
800,0.360543
900,0.353728
1000,0.381481


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 5.2 Train the danger model
Same recipe, 2 buckets (safe / dangerous).

In [24]:
danger_bert = train_bert(train_df.text, train_df.danger, num_labels=2,
                         out_dir="models/danger_bert", epochs=EPOCHS,
                         id2lab={0: "SAFE", 1: "DANGER"})

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/54000 [00:00<?, ? examples/s]

Step,Training Loss
100,0.317850
200,0.109081
300,0.099530
400,0.082287
500,0.073968
600,0.089728
700,0.066398
800,0.076850
900,0.066136
1000,0.051644


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 5.3 Predict with DistilBERT
- `torch.no_grad()` → "just predicting, not learning," saves memory
- `softmax` → turns raw scores into probabilities that add up to 1
- Batches of 64 → faster than one at a time

In [25]:
@torch.no_grad()
def bert_proba(model, texts, batch_size=64):
    model.eval()
    out = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i+batch_size], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(model.device)
        out.append(torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out)

danger_threshold_bert = pick_threshold(val_df.danger, bert_proba(danger_bert, val_df.text)[:, 1])
print("Chosen threshold (DistilBERT):", danger_threshold_bert)

comp_pred_bert   = bert_proba(comp_bert, test_df.text).argmax(1)
danger_prob_bert = bert_proba(danger_bert, test_df.text)[:, 1]

results["DistilBERT (fine-tuned)"] = {
    **score_component(test_df.comp_id, comp_pred_bert),
    **score_danger(test_df.danger, danger_prob_bert, danger_threshold_bert),
    "ms_per_pred": time_per_pred(lambda t: (bert_proba(comp_bert, t), bert_proba(danger_bert, t)), test_df.text),
}

Chosen threshold (DistilBERT): 0.72


# Part 6 — Compare and understand the mistakes

### 6.1 The big comparison table
Write down for each model: **macro_f1**, **pr_auc**, **recall**, **ms_per_pred**.

The question to answer in your write-up: **is the gain worth the extra time and cost?**

Fairness note: DistilBERT runs on a **GPU** here, TF-IDF on a **CPU**. On a CPU-only server, DistilBERT would be much slower. Mention this.

In [26]:
compare = pd.DataFrame(results).T.round(3)
compare

,accuracy,macro_f1,pr_auc,recall,precision,f1,ms_per_pred
dumb baseline,0.234,0.038,0.041,0.000,0.000,0.000,0.000
TF-IDF + LogReg,0.891,0.885,0.830,0.763,0.796,0.779,0.337
DistilBERT (fine-tuned),0.902,0.897,0.876,0.781,0.906,0.839,3.912


### 6.2 Which parts get confused with each other?
A **confusion count**: for each wrong prediction, which real part was mistaken for which. Usually these are parts that really do overlap (e.g., "electrical system" vs "engine").

In [27]:
def top_confusions(y_true, y_pred, n=10):
    pairs = pd.DataFrame({"actual": [id2label[i] for i in y_true],
                          "predicted": [id2label[i] for i in y_pred]})
    wrong = pairs[pairs.actual != pairs.predicted]
    return wrong.value_counts().head(n).rename("count").reset_index()

best_pred = comp_pred_bert if "comp_pred_bert" in globals() else comp_pred_lr
top_confusions(test_df.comp_id, best_pred)

,actual,predicted,count
0,POWER TRAIN,ENGINE,161
1,ENGINE,POWER TRAIN,120
2,ELECTRICAL SYSTEM,ENGINE,99
3,FUEL/PROPULSION SYSTEM,ENGINE,88
4,ENGINE,ELECTRICAL SYSTEM,66
5,ELECTRICAL SYSTEM,STRUCTURE,58
6,ELECTRICAL SYSTEM,POWER TRAIN,56
7,ENGINE,FUEL/PROPULSION SYSTEM,53
8,POWER TRAIN,ELECTRICAL SYSTEM,45
9,STRUCTURE,VISIBILITY/WIPER,39


### 6.3 Read some mistakes yourself
Read 5 wrong predictions. Ask: is the **model** wrong, or is the **label** questionable? Owner-written complaints are messy, and some labels are debatable. This is called **label noise**.

In [28]:
wrong_idx = np.where(test_df.comp_id.values != best_pred)[0][:5]
for i in wrong_idx:
    print(f"ACTUAL: {test_df.component[i]} | PREDICTED: {id2label[best_pred[i]]}")
    print(test_df.text[i][:400], "...\n")

ACTUAL: VISIBILITY/WIPER | PREDICTED: STRUCTURE
panoramic sunroof stuck will not close. generally dealer purportedly recalled select vehicles acknowledging issue, however oem states this particular vin is not included in recall, not making any sense, if they acknowledge problem but decline to resolve for others ...

ACTUAL: STEERING | PREDICTED: ELECTRICAL SYSTEM
when trying to alert a vehicle that was merging into my lane but did not see me, my horn did not work. it randomly does not work at times. every once in a while it will. this poses as a danger and safety risk to drivers and passengers of my 2022 kia carnival for instances as such where another driver may need to be alerted if they donât have safety features for blind spots, backing up, etc. i wo ...

ACTUAL: ELECTRICAL SYSTEM | PREDICTED: ENGINE
the contact owns a 2003 cadillac deville. the contact stated that while driving at an undisclosed speed, the vehicle stalled. the power steering was inoperable. in addition, the cont

### 6.4 Which words drive each prediction?
Logistic regression is **explainable**: each word has a weight per bucket. Show the top words per component. This is how you explain a model to a non-technical stakeholder.

In [29]:
vocab = comp_lr.named_steps["tfidf"].get_feature_names_out()
coefs = comp_lr.named_steps["lr"].coef_

top_words = {id2label[i]: ", ".join(vocab[np.argsort(coefs[i])[::-1][:8]]) for i in range(len(id2label))}
pd.Series(top_words, name="top words")

,top words
AIR BAGS,"airbag, airbags, air, bag, bags, air bag, air bags, deployed"
ELECTRICAL SYSTEM,"electrical, battery, electrical system, radio, system however, screen, alternator, charging"
ENGINE,"engine, oil, the engine, coolant, exhaust, kia, hyundai, stalled"
FORWARD COLLISION AVOIDANCE,"collision, braking, cruise, system, autopilot, phantom, forward collision, brake"
FUEL/PROPULSION SYSTEM,"fuel, gas, tank, pump, gasoline, fuel pump, propulsion, the fuel"
POWER TRAIN,"transmission, train, the transmission, power train, gear, axle, shift, clutch"
SERVICE BRAKES,"brake, brakes, abs, braking, break, the brake, the brakes, breaks"
STEERING,"steering, power steering, the steering, wheel, wobble, steer, rack, power"
STRUCTURE,"door, frame, paint, subframe, structure, hood, rust, latch"
VISIBILITY/WIPER,"windshield, wipers, wiper, visibility, glass, crack, the windshield, window"


# Part 7 — Track experiments with MLflow

**MLflow = a lab notebook that writes itself.** Every run records settings (params) and scores (metrics), so you can compare runs later and never lose a result.

- `sqlite:///mlflow.db` → MLflow keeps its notes in one small database file (newer MLflow no longer allows the old folder style)

Here we log both models. Each new experiment you try (more epochs, different `C`) becomes a new row.

In [30]:
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")   # newer MLflow wants a small database file
mlflow.set_experiment("nhtsa-triage")

def log_run(name, params, metrics):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metrics({k.replace("@", "_at_"): float(v) for k, v in metrics.items()})

log_run("tfidf_logreg", {"model": "tfidf+logreg", "ngram": "1-2", "C": 4.0,
                         "n_train": len(train_df), "threshold": danger_threshold_lr},
        results["TF-IDF + LogReg"])

if "DistilBERT (fine-tuned)" in results:
    log_run("distilbert", {"model": MODEL_NAME, "epochs": EPOCHS, "max_len": MAX_LEN,
                           "n_train": len(train_df), "threshold": danger_threshold_bert},
            results["DistilBERT (fine-tuned)"])

mlflow.search_runs()[["tags.mlflow.runName", "metrics.macro_f1", "metrics.pr_auc",
                      "metrics.recall", "metrics.ms_per_pred"]]

2026/11/14 23:21:44 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/14 23:21:44 INFO mlflow.store.db.utils: Updating database tables
2026/11/14 23:21:46 INFO mlflow.tracking.fluent: Experiment with name 'nhtsa-triage' does not exist. Creating a new experiment.


,tags.mlflow.runName,metrics.macro_f1,metrics.pr_auc,metrics.recall,metrics.ms_per_pred
0,distilbert,0.897039,0.875729,0.781199,3.911882
1,tfidf_logreg,0.884788,0.830048,0.763371,0.337173


### 7.1 Save the models
`joblib` saves the scikit-learn models to a file, like saving a game. The API in Part 10 loads these files.

In [31]:
import joblib
joblib.dump(comp_lr,   "models/comp_lr.joblib")
joblib.dump(danger_lr, "models/danger_lr.joblib")
json.dump({"id2label": id2label, "danger_threshold": danger_threshold_lr},
          open("models/config.json", "w"))
print(os.listdir("models"))

['comp_lr.joblib', 'danger_lr.joblib', 'config.json', 'comp_bert', 'danger_bert']


# Part 8 — Drift: is the world changing?

A model trained on 2021–2023 may slowly get worse as new car models, new features (EVs, driver-assist) and new problems appear. That's **drift**.

We check two things on the **future** data (2025+):
1. **Data drift** with **PSI** (Population Stability Index): did the mix of components or danger scores shift?
   - PSI < 0.10 → stable 🟢 | 0.10–0.25 → watch 🟡 | > 0.25 → alert 🔴
2. **Performance drift:** since NHTSA labels arrive later, we can re-score the model on future data and see if macro-F1 dropped

**Analogy:** a car's dashboard warning lights.

In [32]:
def psi(expected, actual, eps=1e-6):
    expected = np.asarray(expected, float) + eps
    actual   = np.asarray(actual, float) + eps
    expected /= expected.sum(); actual /= actual.sum()
    return float(np.sum((actual - expected) * np.log(actual / expected)))

def binned(values, edges):
    return np.histogram(np.clip(values, edges[0], edges[-1]), bins=edges)[0]

monitor_df = future_df if len(future_df) else test_df
edges = np.linspace(0, 1, 11)

drift = {
    "component mix PSI": psi(train_df.component.value_counts().reindex(TOP_COMPONENTS, fill_value=0),
                             monitor_df.component.value_counts().reindex(TOP_COMPONENTS, fill_value=0)),
    "danger score PSI":  psi(binned(danger_lr.predict_proba(train_df.text)[:, 1], edges),
                             binned(danger_lr.predict_proba(monitor_df.text)[:, 1], edges)),
    "text length PSI":   psi(binned(train_df.text.str.len(), np.linspace(0, 2048, 11)),
                             binned(monitor_df.text.str.len(), np.linspace(0, 2048, 11))),
}
drift["macro_f1 on test"]   = f1_score(test_df.comp_id, comp_lr.predict(test_df.text), average="macro")
drift["macro_f1 on future"] = f1_score(monitor_df.comp_id, comp_lr.predict(monitor_df.text), average="macro")

pd.Series(drift).round(3)

,0
component mix PSI,0.083
danger score PSI,0.027
text length PSI,0.052
macro_f1 on test,0.885
macro_f1 on future,0.873


# Part 9 — Similar complaints + emerging-issues report

### 9.1 Similar-complaint search
Same idea as your PubMedQA project: turn complaints into **meaning addresses** (embeddings), store them in **Chroma**, and find the nearest ones.

Use case: an engineer reads one scary complaint and asks, *"Have others seen this?"*

- `all-MiniLM-L6-v2` → small, fast embedding model (reads up to 256 tokens)
- **Metadata** (make, model, year, component) lets us **filter**, e.g. only FORD

In [33]:
from sentence_transformers import SentenceTransformer
import chromadb

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
search_pool = pd.concat([test_df, future_df], ignore_index=True)

chroma = chromadb.Client()
try:
    chroma.delete_collection("complaints")
except Exception:
    pass
col = chroma.create_collection("complaints", metadata={"hnsw:space": "cosine"})

embs = embedder.encode(search_pool.text.tolist(), normalize_embeddings=True,
                       batch_size=128, show_progress_bar=True)
B = 2000
for i in range(0, len(search_pool), B):
    part = search_pool.iloc[i:i+B]
    col.add(ids=part.ODINO.astype(str).tolist(), documents=part.text.tolist(),
            embeddings=embs[i:i+B].tolist(),
            metadatas=[{"make": str(r.MAKETXT), "model": str(r.MODELTXT), "year": str(r.YEARTXT),
                        "component": r.component, "date": str(r.date.date()), "danger": int(r.danger)}
                       for r in part.itertuples()])
print("Stored:", col.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/235 [00:00<?, ?it/s]

Stored: 30000


In [34]:
def similar_complaints(text, k=5, make=None):
    q = embedder.encode([text.lower()], normalize_embeddings=True).tolist()
    where = {"make": make.upper()} if make else None
    res = col.query(query_embeddings=q, n_results=k, where=where)
    rows = []
    for oid, meta, doc, dist in zip(res["ids"][0], res["metadatas"][0], res["documents"][0], res["distances"][0]):
        rows.append({"odino": oid, "similarity": round(1 - dist, 3), **meta, "text": doc[:200] + "..."})
    return pd.DataFrame(rows)

similar_complaints("while driving on the highway the engine suddenly lost power and the dashboard lights flickered")

,odino,similarity,year,model,date,danger,component,make,text
0,11656460,0.655,2017,MDX,2025-04-23,0,ENGINE,ACURA,"while driving on the interstate, the engine suddenly lost power and a lot of malfunction indicator lights turned on. driver was able to pull over without incident. mechanic reported the engine had..."
1,11612871,0.645,2024,XT6,2024-09-05,0,POWER TRAIN,CADILLAC,"the vehicle lost power while driving and could not accelerate. fortunately, it was on a residential street and not a highway or busy road...."
2,11740028,0.617,2017,PILOT,2026-05-26,0,ENGINE,HONDA,while driving on the highway at approx 60mph my car completely lost power. i had no warning lights on the dashboard and the only warning i got was an alarm sound right as all the power to the vehi...
3,11622116,0.616,2025,EQUINOX,2024-10-26,0,ELECTRICAL SYSTEM,CHEVROLET,was driving down the road with a sudden loss of power to the infotainment system plus dashboard resulting in all black and being unable to see your speed/turn signals. the steering wheel lights su...
4,11749941,0.615,2026,SIERRA 1500,2026-07-13,0,ENGINE,GMC,while driving vehicle the engine lost power and shut off in traffic. there were no warning signs of issues at that time. the dashboard went blank and then reappeared and the truck was in park. aft...


### 9.2 Find spikes (emerging issues)
A **spike** = a make + model + component with many more complaints this month than usual.

Rule (simple and explainable):
- Count complaints per month for each make + model + component
- Compare the **last full month** with the **average of the 3 months before**
- Spike if **≥ 10 complaints** and **≥ 2× the usual**

**Analogy:** a teacher noticing 10 kids got stomach aches today when usually it's 2.

We use **all** clean complaints here (not the sample), because counting needs every complaint.

In [35]:
def find_spikes(d, min_count=10, min_ratio=2.0):
    d = d.copy()
    d["month"] = d.date.dt.to_period("M")
    target = d.month.max() - 1                       # last FULL month (latest may be partial)
    prev = [target - 3, target - 2, target - 1]
    counts = (d[d.month.isin(prev + [target])]
                .groupby(["MAKETXT", "MODELTXT", "component", "month"]).size()
                .unstack("month", fill_value=0)
                .reindex(columns=prev + [target], fill_value=0))
    out = pd.DataFrame({"this_month": counts[target],
                        "usual": counts[prev].mean(axis=1).round(1)})
    out["ratio"] = (out.this_month / out.usual.clip(lower=1)).round(1)
    spikes = out[(out.this_month >= min_count) & (out.ratio >= min_ratio)]
    return spikes.sort_values("ratio", ascending=False).reset_index(), target

spikes, spike_month = find_spikes(clean)
print("Month checked:", spike_month)
spikes.head(10)

Month checked: 2026-10


,MAKETXT,MODELTXT,component,this_month,usual,ratio
0,MAZDA,CX-90,STEERING,18,7.7,2.3
1,CHEVROLET,COLORADO,STEERING,10,4.7,2.1
2,FORD,EXPLORER,ENGINE,10,4.7,2.1
3,HONDA,RIDGELINE,ENGINE,13,6.3,2.1
4,FORD,TAURUS,BACK OVER PREVENTION,16,8.0,2.0


### 9.3 Load a small LLM to write the report
We use **Qwen2.5-1.5B-Instruct** (no access request needed, fits a T4).

Same rule as PubMedQA: **only use the complaints we give you.** This is called **grounding**, and it stops the model from making things up.

In [36]:
from transformers import pipeline
llm = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct",
               torch_dtype=torch.float16, device_map="auto")

SYSTEM = ("You are a vehicle safety analyst. Use ONLY the complaints provided. "
          "Do not guess causes that the complaints do not mention. "
          "If the complaints are too unclear, say 'Not enough information'.")

def summarize_spike(make, model, component, n=6):
    rows = clean[(clean.MAKETXT == make) & (clean.MODELTXT == model) & (clean.component == component)]
    rows = rows.sort_values("date", ascending=False).head(n)
    ctx = "\n\n".join(f"[{i+1}] {t[:600]}" for i, t in enumerate(rows.text))
    user = (f"Complaints about {make} {model}, component: {component}\n\n{ctx}\n\n"
            "In 3 short bullet points: what fails, when it happens, and the safety risk.")
    out = llm([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
              max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][-1]["content"].strip(), rows.ODINO.tolist()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

### 9.4 The weekly report
For the top 3 spikes, write a short summary **plus the complaint IDs used**, so an engineer can check the evidence.

In [37]:
report_parts = [f"# Emerging issues — {spike_month}\n"]
for r in spikes.head(3).itertuples():
    summary, ids = summarize_spike(r.MAKETXT, r.MODELTXT, r.component)
    report_parts.append(
        f"## {r.MAKETXT} {r.MODELTXT} — {r.component}\n"
        f"**{r.this_month} complaints** this month vs **{r.usual}** usual ({r.ratio}×)\n\n"
        f"{summary}\n\n_Evidence: NHTSA complaint IDs {', '.join(map(str, ids))}_\n")
if len(spikes) == 0:
    report_parts.append("No spikes met the threshold this month.")

report_md = "\n".join(report_parts)
open("emerging_issues.md", "w").write(report_md)
print(report_md)

# Emerging issues — 2026-10

## MAZDA CX-90 — STEERING
**18 complaints** this month vs **7.7** usual (2.3×)

**What fails:** The steering wheel becomes sticky and difficult to control during turns.
  
**When it happens:** During turns, particularly when making sharp left turns at moderate speeds.

**Safety Risk:** There is an increased risk of accidents due to reduced control of the vehicle during turns, which could lead to loss of control and potential collisions.

_Evidence: NHTSA complaint IDs 11777422, 11777357, 11776580, 11776549, 11774157, 11773702_

## CHEVROLET COLORADO — STEERING
**10 complaints** this month vs **4.7** usual (2.1×)

The Chevrolet Colorado's steering system experiences failures under various conditions:

- **When:** The steering system fails during acceleration, deceleration, turns, and while driving.
- **Safety Risk:** Loss of vehicle control, increased risk of accidents, especially at high speeds or when turning, which poses a significant safety hazard.

_Evi

# Part 10 — Serve the model as an API

An **API** is a vending machine for your model: send a complaint in, get predictions out. Any app (a dashboard, a website, Power BI) can use it.

We serve the **TF-IDF models**: they run on a cheap CPU server with no GPU. That's a real deployment trade-off you can explain in interviews. Swapping in DistilBERT is a small change.

`%%writefile` saves the cell as a file instead of running it.

In [38]:
%%writefile app_api.py
import json, joblib
from fastapi import FastAPI
from pydantic import BaseModel

comp_lr   = joblib.load("models/comp_lr.joblib")
danger_lr = joblib.load("models/danger_lr.joblib")
cfg       = json.load(open("models/config.json"))
id2label  = {int(k): v for k, v in cfg["id2label"].items()}
THRESHOLD = cfg["danger_threshold"]

app = FastAPI(title="NHTSA Complaint Triage")

class Complaint(BaseModel):
    text: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(c: Complaint):
    text = c.text.lower()
    probs = comp_lr.predict_proba([text])[0]
    top3 = probs.argsort()[::-1][:3]
    danger = float(danger_lr.predict_proba([text])[0, 1])
    return {
        "component": id2label[int(comp_lr.classes_[top3[0]])],
        "top3": [{"component": id2label[int(comp_lr.classes_[i])], "prob": round(float(probs[i]), 3)} for i in top3],
        "danger_score": round(danger, 3),
        "danger_flag": danger >= THRESHOLD,
    }

Writing app_api.py


### 10.1 Start the API and test it
- `subprocess.Popen` starts the server **in the background** inside Colab
- `requests.post` acts like a customer sending a complaint

In [39]:
import subprocess, requests
server = subprocess.Popen(["uvicorn", "app_api:app", "--port", "8000"])
time.sleep(6)

print(requests.get("http://127.0.0.1:8000/health").json())
print(requests.post("http://127.0.0.1:8000/predict",
                    json={"text": "The airbag light stays on and the passenger airbag did not deploy in a crash"}).json())

{'status': 'ok'}
{'component': 'AIR BAGS', 'top3': [{'component': 'AIR BAGS', 'prob': 1.0}, {'component': 'STRUCTURE', 'prob': 0.0}, {'component': 'SERVICE BRAKES', 'prob': 0.0}], 'danger_score': 0.857, 'danger_flag': True}


### 10.2 Package it with Docker
**Docker = a lunchbox** that holds your code, models and libraries together, so it runs the same on any computer.

Colab can't run Docker, so these files are for your **laptop or a cloud server**. Commands to run there:
```
docker build -t nhtsa-triage .
docker run -p 8000:8000 nhtsa-triage
```
Then open `http://localhost:8000/docs` to see an automatic test page FastAPI builds for you.

In [40]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
joblib
pydantic

Writing requirements.txt


In [41]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app_api.py .
COPY models/comp_lr.joblib models/danger_lr.joblib models/config.json models/
EXPOSE 8000
CMD ["uvicorn", "app_api:app", "--host", "0.0.0.0", "--port", "8000"]

Writing Dockerfile


> Tip: train the models with the **same scikit-learn version** you put in `requirements.txt`, or loading the `.joblib` files may fail. Pin it with `scikit-learn==<version>` after checking `sklearn.__version__`.

# Part 11 — Demo app (Gradio)

Three tabs a "customer" (a safety engineer) can click:
1. **Triage** → paste a complaint, see the part and danger score
2. **Similar** → find past complaints like it, optionally for one make
3. **Emerging issues** → this month's report

`share=True` gives a temporary public link. For a permanent link, upload to **Hugging Face Spaces**.

In [42]:
import gradio as gr

def ui_triage(text):
    t = text.lower()
    probs = comp_lr.predict_proba([t])[0]
    comp = {id2label[int(c)]: float(p) for c, p in zip(comp_lr.classes_, probs)}
    danger = float(danger_lr.predict_proba([t])[0, 1])
    flag = "🔴 FLAG FOR REVIEW" if danger >= danger_threshold_lr else "🟢 Routine"
    return comp, f"{flag} (danger score {danger:.2f})"

def ui_similar(text, make):
    return similar_complaints(text, k=5, make=make or None)

with gr.Blocks(title="NHTSA Complaint Triage") as app:
    gr.Markdown("# Vehicle Safety Complaint Triage")
    with gr.Tab("Triage"):
        inp = gr.Textbox(label="Complaint text", lines=4)
        btn = gr.Button("Triage")
        out_comp = gr.Label(label="Component", num_top_classes=3)
        out_danger = gr.Textbox(label="Danger")
        btn.click(ui_triage, inp, [out_comp, out_danger])
    with gr.Tab("Similar"):
        s_inp = gr.Textbox(label="Complaint text", lines=3)
        s_make = gr.Textbox(label="Make (optional, e.g. FORD)")
        s_btn = gr.Button("Search")
        s_out = gr.Dataframe()
        s_btn.click(ui_similar, [s_inp, s_make], s_out)
    with gr.Tab("Emerging issues"):
        gr.Markdown(report_md)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f439bf72f3dc201222.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Part 12 — Save results and write-up

In [43]:
compare.to_csv("model_comparison.csv")
pd.Series(drift).to_csv("drift.csv")
spikes.to_csv("spikes.csv", index=False)
# Also save to Drive (needs Part 0.4):
# for f in ["model_comparison.csv", "drift.csv", "spikes.csv", "emerging_issues.md"]:
#     !cp {f} {SAVE_DIR}/

## Fill in your numbers
- Danger rate (Part 2.2): ___
- Component macro-F1: dumb ___ / TF-IDF ___ / DistilBERT ___
- Danger PR-AUC: dumb ___ / TF-IDF ___ / DistilBERT ___
- Danger recall at chosen threshold: TF-IDF ___ / DistilBERT ___
- ms per prediction: TF-IDF ___ / DistilBERT ___
- Drift PSI and macro-F1 on future vs test (Part 8): ___

## README structure
1. **Problem:** thousands of complaints a month; engineers can't read them all
2. **Question:** is a fine-tuned transformer worth its cost over a simple model?
3. **Data:** NHTSA complaints, time-based split, privacy columns dropped
4. **Method:** TF-IDF + LogReg vs DistilBERT → error analysis → MLflow → search → spike report → drift → API → app
5. **Results:** comparison table + speed
6. **Deployment choice:** which model you'd ship and why
7. **Limitations:** owner-written text, label noise, single-component complaints only, top 10 parts only, US data, one seed, threshold picked on a random validation slice
8. **Next steps**

## Experiments to try next (pick 1–2)
- Epochs 1 vs 2 vs 3 for DistilBERT
- `MAX_LEN` 128 vs 256 vs 512 (speed vs accuracy)
- Train size 10k vs 30k vs 60k (does more data help the simple model more?)
- Multi-label: predict **all** components a complaint mentions
- Link spikes to actual **recalls** (NHTSA recall flat file) and check how early you'd have caught them

## Resume bullets (fill in real numbers)
- Built an NLP triage system for NHTSA vehicle safety complaints that classifies the failed component and flags crash/fire/injury risk, using a time-based split to avoid leakage
- Fine-tuned DistilBERT and benchmarked it against a TF-IDF + logistic regression baseline (macro-F1 ___ vs ___, ___ ms vs ___ ms per prediction)
- Deployed the model as a Dockerized FastAPI service with MLflow tracking, PSI drift monitoring, embedding-based similar-complaint search, and an LLM-generated emerging-issues report

## Common errors
| Error | Fix |
|---|---|
| Download fails / 404 | File names changed; download the complaints zip from NHTSA's flat-files page and upload to `data/` |
| `MemoryError` reading files | Use only the 2020–2024 zip, or lower `N_TRAIN` |
| `CUDA out of memory` | Lower `per_device_train_batch_size` to 16 or `MAX_LEN` to 128 |
| Empty `future_df` | No 2025+ file; Part 8 falls back to test data automatically |
| No spikes found | Lower `min_count` or `min_ratio` in `find_spikes` |
| Port 8000 already in use | Run `server.kill()` then start again |
| `NameError` | A cell above was skipped; run all cells in order |